In [ ]:

import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

In [ ]:
# accessing parameters
# 可以看到所有信息
print(net[2].state_dict())
# 为了得到具体的参数
print(net[2].bias.data)
net.state_dict()['2.bias'].data
print(net[2].weight.grad)
# *可以将【】中的值提取出来
print(*[(name,param.shape) for name, param in net.named_parameters()])


In [ ]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # add_module is a pytorch method which operate on dic 'self._module'
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

# onion-like
rgnet[0][1][0].bias.data

# initialize parameters

In [ ]:
# built in

def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,0,1)
        nn.init.zeros_(m.bias)
net.apply(init_normal)

In [ ]:
# constant

def init_constant(m,val):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, val)
        nn.init.zeros_(m.bias)
net.apply(init_constant)    

In [ ]:
# xavier

def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)

In [ ]:
# self-designed

# 1/4 5~10 1/2 0 1/4 -10~-5

def my_init(m):
    if type(m) == nn.Linear:
        nn.Linear.uniform_(m.weight,0, 10)
    # m.weight *= m.weight.abs() >= 5
    m.weight.data *= m.weight.abs() >= 5

In [ ]:
# change directly
# remember to add data

net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42

In [ ]:
# parameter bonding/weight sharing
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)

# reduce the number of trainable parameters and enforce feature mapping
print(net[2].weight.data[0] == net[4].weight.data[0])